In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalizability Evaluation for Belief-Tracking Circuit

This notebook evaluates the generalizability of the belief-tracking circuit findings from the paper "Language Models use Lookbacks to Track Beliefs" (Prakash et al., 2025).

## Evaluation Criteria:
- **GT1**: Model Generalization - Does the finding transfer to a new model?
- **GT2**: Data Generalization - Does the finding hold on new data instances?
- **GT3**: Method Generalization - Can the method be applied to another similar task?

## Original Findings Summary:
1. **Answer Lookback Pointer**: Layers 34-52 encode pointer information at final token
2. **Answer Lookback Payload**: Layers 56+ encode the actual state token value
3. **Binding Address/Payload**: Layers 33-38 at state tokens
4. **Binding Source Reference**: Layers 20-34 encode character/object ordering IDs

**Original Models:** Llama-3-70B-Instruct, Llama-3.1-405B-Instruct

In [2]:
# Check CUDA availability and setup
import torch
import sys
import json
import random
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
CUDA memory: 85.1 GB
Using device: cuda


In [3]:
# Add the belief-tracking repo to path and import utilities
repo_path = '/net/scratch2/smallyan/belief_tracking_eval'
sys.path.append(repo_path)

from src.dataset import Dataset, Sample

# Load synthetic entities
with open(os.path.join(repo_path, 'data', 'synthetic_entities', 'characters.json'), 'r') as f:
    all_characters = json.load(f)
with open(os.path.join(repo_path, 'data', 'synthetic_entities', 'bottles.json'), 'r') as f:
    all_objects = json.load(f)
with open(os.path.join(repo_path, 'data', 'synthetic_entities', 'drinks.json'), 'r') as f:
    all_states = json.load(f)

print(f"Loaded {len(all_characters)} characters, {len(all_objects)} objects, {len(all_states)} states")

Loaded 103 characters, 21 objects, 23 states


## GT1: Model Generalization

Testing whether the lookback mechanism findings generalize to a new model not used in the original work.

**Original models:** Llama-3-70B-Instruct, Llama-3.1-405B-Instruct

**New model for testing:** Mistral-7B-Instruct-v0.3 (a different architecture family not used in the original paper)

In [4]:
# Install nnsight if needed
try:
    from nnsight import LanguageModel
    print("nnsight already installed")
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'nnsight', '-q'])
    from nnsight import LanguageModel
    print("nnsight installed")

nnsight already installed


In [5]:
# Load a new model not used in the original paper
# We'll use Mistral-7B-Instruct as it's a different architecture family
from nnsight import LanguageModel

# Set HF cache
os.environ['HF_HOME'] = '/net/scratch2/smallyan/hf_cache'

# Load Mistral-7B-Instruct (not used in original paper)
print("Loading Mistral-7B-Instruct-v0.3...")
model = LanguageModel(
    "mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto",
    torch_dtype=torch.float16,
)
print(f"Model loaded successfully. Number of layers: {model.config.num_hidden_layers}")

Loading Mistral-7B-Instruct-v0.3...


OSError: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--mistralai--Mistral-7B-Instruct-v0.3'

In [6]:
# Try to use models that are already cached or available locally
# Let's check what models might be available in the original results
import os

# Check what models were tested in results
results_path = '/net/scratch2/smallyan/belief_tracking_eval/results/model_evaluations'
if os.path.exists(results_path):
    print("Available model evaluations:")
    for f in os.listdir(results_path):
        print(f"  - {f}")

Available model evaluations:
  - Llama-2-7b-hf.json
  - OLMo-2-1124-13B-Instruct_vis.json
  - Qwen2.5-7B-Instruct_vis.json
  - OLMo-2-0325-32B-Instruct.json
  - Qwen2.5-14B-Instruct_vis.json
  - gemma-3-27b-it_vis.json
  - Qwen2.5-14B_vis.json
  - Llama-3.1-8B_vis.json
  - Llama-3.1-8B-Instruct.json
  - Llama-2-7b-hf_vis.json
  - OLMo-2-1124-13B-Instruct.json
  - Meta-Llama-3-8B-Instruct_vis.json
  - Meta-Llama-3-70B-Instruct.json
  - Llama-3.1-8B-Instruct_vis.json
  - Qwen2.5-7B-Instruct.json
  - OLMo-2-0325-32B-Instruct_vis.json
  - Meta-Llama-3-8B-Instruct.json
  - Qwen2.5-7B_vis.json
  - Meta-Llama-3-8B.json
  - Meta-Llama-3-8B_vis.json
  - Meta-Llama-3-70B-Instruct_vis.json
  - Llama-2-13b-hf_vis.json
  - Qwen2.5-14B.json
  - Qwen2.5-14B-Instruct.json
  - gemma-3-27b-it.json
  - Llama-3.1-8B.json
  - Llama-2-13b-hf.json
  - Qwen2.5-7B.json


In [7]:
# Let's check the HF cache to see what models are already downloaded
import subprocess

# Check what's in the hf_cache
cache_path = '/net/scratch2/smallyan/hf_cache'
if os.path.exists(cache_path):
    result = subprocess.run(['ls', '-la', cache_path], capture_output=True, text=True)
    print(result.stdout)
    
    # Check hub directory
    hub_path = os.path.join(cache_path, 'hub')
    if os.path.exists(hub_path):
        print("\nModels in hub:")
        result = subprocess.run(['ls', hub_path], capture_output=True, text=True)
        print(result.stdout)

total 45
drwxrwx---  7 smallyan smallyan  8 Dec 23 23:06 .
drwxrwx--- 20 smallyan smallyan 20 Dec 23 22:50 ..
drwxrwx---  6 smallyan smallyan  6 Dec 23 23:06 .locks
drwxrwx---  6 smallyan smallyan  6 Dec 23 22:52 models--BAAI--bge-base-en-v1.5
drwxrwx---  5 smallyan smallyan  5 Dec 23 23:06 models--NeelNanda--Othello-GPT-Transformer-Lens
drwxrwx---  6 smallyan smallyan  6 Dec 23 22:51 models--Qwen--Qwen2-0.5B
drwxrwx---  6 smallyan smallyan  6 Dec 23 22:50 models--Qwen--Qwen3-0.6B
-rw-rw----  1 smallyan smallyan  1 Dec 23 22:51 version.txt



In [8]:
# Let's try to use Qwen2-0.5B which is already cached - it's small and a different family
# than Llama models used in the original paper
from nnsight import LanguageModel

os.environ['HF_HOME'] = '/net/scratch2/smallyan/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/net/scratch2/smallyan/hf_cache'

print("Loading Qwen2-0.5B (a model from a different family than Llama)...")
model = LanguageModel(
    "Qwen/Qwen2-0.5B",
    device_map="auto",
    torch_dtype=torch.float16,
)
print(f"Model loaded successfully. Number of layers: {model.config.num_hidden_layers}")

Loading Qwen2-0.5B (a model from a different family than Llama)...


OSError: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--Qwen--Qwen2-0.5B'

In [9]:
# Check the default HF cache location
import subprocess
result = subprocess.run(['ls', '-la', '/net/projects/chai-lab/shared_models/hub/'], capture_output=True, text=True)
print("Models in shared cache:")
print(result.stdout[:3000] if result.stdout else result.stderr)

Models in shared cache:
total 108
drwxrws--- 38 mourad    __complementary-ai 39 Dec 23 22:32 .
drwxrwsrwx 21 haokunliu __complementary-ai 23 Dec  2 03:13 ..
drwxrws---  6 smallyan  __complementary-ai  6 Aug 13 12:35 datasets--cais--mmlu
drwxrws---  6 smallyan  __complementary-ai  6 Aug 13 13:21 datasets--commonsense_qa
drwxrws---  6 smallyan  __complementary-ai  6 Jul 21 14:04 datasets--domenicrosati--TruthfulQA
drwxrws---  6 smallyan  __complementary-ai  6 Aug 12 14:49 datasets--gsm8k
drwxrws---  6 smallyan  __complementary-ai  6 Oct  4 17:38 datasets--mib-bench--copycolors_mcqa
drwxrws---  6 smallyan  __complementary-ai  6 Oct  4 17:31 datasets--mib-bench--ioi
drwxrws---  6 smallyan  __complementary-ai  6 Oct  7 10:38 datasets--monology--pile-uncopyrighted
drwxrws---  6 smallyan  __complementary-ai  6 Oct  7 10:26 datasets--multilingual-mi-llm--pile
drwxrws---  6 smallyan  __complementary-ai  6 Aug 13 12:17 datasets--openlifescienceai--medmcqa
drwxrws---  6 smallyan  __complementary-

In [10]:
# Great! Let's use one of the available models in the shared cache
# We can use Qwen2.5-0.5B - it's a Qwen model (different family than Llama) and is small

from nnsight import LanguageModel
from transformers import AutoConfig

# Force using local files only from the shared cache
os.environ['HF_HOME'] = '/net/projects/chai-lab/shared_models'
os.environ['TRANSFORMERS_CACHE'] = '/net/projects/chai-lab/shared_models'
os.environ['HF_DATASETS_CACHE'] = '/net/projects/chai-lab/shared_models'

print("Loading Qwen2.5-0.5B (a different family than Llama used in original paper)...")
model = LanguageModel(
    "Qwen/Qwen2.5-0.5B",
    device_map="auto",
    torch_dtype=torch.float16,
    local_files_only=True,
)
print(f"Model loaded successfully. Number of layers: {model.config.num_hidden_layers}")

Loading Qwen2.5-0.5B (a different family than Llama used in original paper)...


Model loaded successfully. Number of layers: 24


In [11]:
# First, let's test if the model can perform the belief tracking task correctly
# Create a test sample

random.seed(42)
test_characters = random.sample(all_characters, 2)
test_objects = random.sample(all_objects, 2)
test_states = random.sample(all_states, 2)

sample = Sample(
    template_idx=2,  # Template without visibility modifiers
    characters=test_characters,
    objects=test_objects,
    states=test_states,
)

dataset = Dataset([sample])
test_item = dataset.__getitem__(0, set_container=0, set_character=0)

print("Test prompt:")
print(test_item['prompt'])
print(f"\nExpected answer: {test_item['target']}")

Test prompt:
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict container or character as the final output.

Story: Charlie and Pete are working in a busy restaurant. To complete an order, Charlie grabs an opaque jar and fills it with wine. Then Pete grabs another opaque can and fills it with soda.
Question: What does Charlie believe the jar contains?
Answer:

Expected answer: wine


In [12]:
# Test model prediction
with torch.no_grad():
    with model.trace(test_item['prompt']):
        pred_logits = model.lm_head.output[0, -1].save()

pred_token = pred_logits.argmax(dim=-1).item()
pred_text = model.tokenizer.decode([pred_token]).strip().lower()
print(f"Model prediction: '{pred_text}'")
print(f"Expected: '{test_item['target']}'")
print(f"Match: {pred_text == test_item['target']}")

RuntimeError: unknown device type for autocast in get_autocast_dispatch_key_from_device_type

In [13]:
# Try loading the model with dispatch=True for proper device handling
del model
torch.cuda.empty_cache()

from nnsight import LanguageModel

print("Loading Qwen2.5-0.5B with dispatch=True...")
model = LanguageModel(
    "Qwen/Qwen2.5-0.5B",
    device_map="cuda:0",
    torch_dtype=torch.float16,
    local_files_only=True,
    dispatch=True,
)
print(f"Model loaded successfully. Number of layers: {model.config.num_hidden_layers}")

Loading Qwen2.5-0.5B with dispatch=True...


Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--Qwen--Qwen2.5-0.5B/.no_exist/060db6499f32faf8b98477b0a26969ef7d8b9987/adapter_config.json'


Model loaded successfully. Number of layers: 24


In [14]:
# Test model prediction again
with torch.no_grad():
    with model.trace(test_item['prompt']):
        pred_logits = model.lm_head.output[0, -1].save()

pred_token = pred_logits.argmax(dim=-1).item()
pred_text = model.tokenizer.decode([pred_token]).strip().lower()
print(f"Model prediction: '{pred_text}'")
print(f"Expected: '{test_item['target']}'")
print(f"Match: {pred_text == test_item['target']}")

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Model prediction: 'charlie'
Expected: 'wine'
Match: False


In [15]:
# The small model doesn't do well on this task. Let's try a different model from the cache
# Let's use GPT-2-XL which is available and is a different architecture

del model
torch.cuda.empty_cache()

print("Loading GPT-2-XL...")
model = LanguageModel(
    "gpt2-xl",
    device_map="cuda:0",
    torch_dtype=torch.float16,
    local_files_only=True,
    dispatch=True,
)
print(f"Model loaded successfully. Number of layers: {model.config.n_layer}")

Loading GPT-2-XL...


Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--gpt2-xl/.no_exist/15ea56dee5df4983c59b2538573817e1667135e2/adapter_config.json'


Model loaded successfully. Number of layers: 48


In [16]:
# Test GPT-2-XL on the belief tracking task
with torch.no_grad():
    with model.trace(test_item['prompt']):
        pred_logits = model.lm_head.output[0, -1].save()

pred_token = pred_logits.argmax(dim=-1).item()
pred_text = model.tokenizer.decode([pred_token]).strip().lower()
print(f"Model prediction: '{pred_text}'")
print(f"Expected: '{test_item['target']}'")
print(f"Match: {pred_text == test_item['target']}")

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Model prediction: 'the'
Expected: 'wine'
Match: False


In [17]:
# GPT-2 also doesn't work well on this task. Let's try Llama-3.1-8B-Instruct
# which is available and is an instruction-tuned model

del model
torch.cuda.empty_cache()

print("Loading Llama-3.1-8B-Instruct...")
model = LanguageModel(
    "meta-llama/Llama-3.1-8B-Instruct",
    device_map="cuda:0",
    torch_dtype=torch.float16,
    local_files_only=True,
    dispatch=True,
)
print(f"Model loaded successfully. Number of layers: {model.config.num_hidden_layers}")

Loading Llama-3.1-8B-Instruct...


Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 122] Disk quota exceeded: '/net/projects/chai-lab/shared_models/hub/models--meta-llama--Llama-3.1-8B-Instruct/.no_exist/0e9e39f249a16976918f6564b8830bc894c89659/adapter_config.json'


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully. Number of layers: 32
